# [실습 2] LoRA 방식 PEFT 어댑터 구현

## 실습 개요

> "전체 모델을 업데이트하지 않고도 태스크에 맞는 파라미터를 추가할 수 있을까?"

이번 실습에서는 작은 선형 분류 모델에 LoRA 스타일의 저랭크 어댑터를 직접 붙입니다. PEFT의 핵심인 base weight 동결, 작은 trainable adapter 추가, 학습 가능한 파라미터 비율 계산을 코드로 확인합니다.

실제 대형 언어 모델에서는 `peft` 라이브러리를 사용하는 경우가 많지만, 내부 원리를 한 번 구현해 보면 어떤 파라미터가 학습되고 어떤 파라미터가 고정되는지 더 명확해집니다.

## 실습 목표

1. 일반 Linear 레이어와 LoRA Linear 레이어의 구조를 비교한다.
2. base weight를 동결하고 adapter 파라미터만 학습 대상으로 남긴다.
3. 학습 가능한 파라미터 비율을 계산한다.
4. 작은 데이터에서 adapter만 업데이트되는지 확인한다.
5. [TODO] 코드를 완성해 LoRA forward 연산을 구현한다.

## 데이터 출처

- 데이터: 실습용 문장 임베딩 벡터 직접 생성
- 설정 파일: `config/lora_config.json`

---
## 1. 라이브러리와 설정 준비

### LoRA 설정값 읽기

LoRA는 rank, alpha, dropout처럼 작은 설정값 몇 개로 adapter의 표현력과 학습 안정성이 달라집니다. rank가 커지면 더 많은 변화를 표현할 수 있지만, 학습 파라미터와 저장 비용도 함께 늘어납니다.

설정을 파일로 남겨 두면 같은 base 모델 위에 어떤 adapter를 붙였는지 추적하기 쉽습니다. 여러 태스크용 adapter를 실험할 때는 코드보다 설정 파일이 실험의 이름표 역할을 합니다.

In [1]:
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

config = json.loads(Path('config/lora_config.json').read_text(encoding='utf-8'))
torch.manual_seed(config['seed'])
print('LoRA 실습 설정:', config)

LoRA 실습 설정: {'seed': 7, 'input_dim': 24, 'num_labels': 3, 'num_samples': 96, 'rank': 4, 'alpha': 8, 'dropout': 0.05, 'learning_rate': 0.05, 'epochs': 8}


### rank에 따른 학습 파라미터 규모 예상하기

LoRA의 rank를 정하기 전에 파라미터 수가 어떻게 늘어나는지 먼저 계산해 봅니다. 선형 레이어 하나만 보더라도 rank가 커질수록 `A`와 `B` 행렬의 크기가 함께 증가합니다.

이런 계산은 대형 모델에서 더 중요합니다. rank를 조금만 올려도 모든 target module에 adapter가 붙으면 전체 trainable parameter가 크게 늘어날 수 있습니다. 실습에서는 작은 숫자로 보지만, 실제 프로젝트에서는 성능 개선 폭과 학습 비용을 함께 놓고 rank를 결정합니다.

In [ ]:
for rank_candidate in [1, 2, 4, 8]:
    adapter_params = (config['input_dim'] + config['num_labels']) * rank_candidate 
    print(f'rank={rank_candidate}일 때 adapter 파라미터 수:', adapter_params)

rank=1일 때 adapter 파라미터 수: 27
rank=2일 때 adapter 파라미터 수: 54
rank=4일 때 adapter 파라미터 수: 108
rank=8일 때 adapter 파라미터 수: 216


---
## 2. 실습용 데이터 구성

### 문장 임베딩을 가정한 벡터 데이터 만들기

Transformer 전체를 돌리지 않고, 이미 추출된 문장 임베딩이라고 가정한 벡터 데이터를 사용합니다. 이렇게 하면 토크나이징이나 attention 계산보다 LoRA 레이어의 구조와 학습 대상에 집중할 수 있습니다.

각 샘플은 `input_dim` 크기의 벡터이고 라벨은 세 개 클래스 중 하나입니다. 실제 문장 분류에서는 Transformer backbone이 이 벡터를 만들고, 분류 head나 adapter가 태스크별 출력을 조정합니다.

In [ ]:
def make_embedding_dataset(n=96, input_dim=24, num_labels=3):
    x = torch.randn(n, input_dim)
    projection = torch.randn(input_dim, num_labels)
    y = torch.argmax(x @ projection, dim=-1)
    return x, y

x_train, y_train = make_embedding_dataset(
    n=config['num_samples'],
    input_dim=config['input_dim'],
    num_labels=config['num_labels'],
)
print('학습 입력과 라벨 크기:', x_train.shape, y_train.shape)
print('라벨별 샘플 수:', torch.bincount(y_train, minlength=config['num_labels']).tolist())

학습 입력과 라벨 크기: torch.Size([96, 24]) torch.Size([96])
라벨별 샘플 수: [44, 23, 29]


---
## 3. [TODO] LoRA Linear 레이어 구현

### [TODO] base weight와 adapter update 분리하기 !!!!@@@!@!@!@

LoRA는 기존 Linear weight를 직접 바꾸지 않고, 작은 두 행렬 `A`, `B`를 통과한 값을 추가 update로 더합니다. base weight는 고정되어 있으므로 원본 모델을 유지한 채 태스크별 변화만 adapter에 저장할 수 있습니다.

forward 흐름은 세 부분으로 나누어 보면 이해하기 쉽습니다. 먼저 base Linear가 원래 출력을 만들고, 입력에 dropout을 적용한 뒤 `lora_a`, `lora_b`를 차례로 통과시켜 저랭크 update를 만듭니다. 마지막으로 scaling을 곱해 update 크기를 조절하고 base 출력에 더합니다.

[TODO]에서는 이 adapter branch를 완성합니다. 텐서 shape은 `x -> rank -> out_features` 순서로 바뀌어야 하며, 마지막 출력은 base Linear 출력과 같은 shape이어야 더할 수 있습니다.

완성해야 할 변수는 다음과 같습니다.

- `dropped_x`: 입력 `x`에 `self.dropout`을 적용한 값입니다.
- `low_rank_features`: `self.lora_a(dropped_x)`의 결과입니다.
- `adapter_logits`: `self.lora_b(low_rank_features)`의 결과입니다.
- `adapter_output`: `adapter_logits * self.scaling`으로 크기를 조절한 adapter update입니다.

In [6]:
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=4, alpha=8, dropout=0.0):
        super().__init__()
        self.base = nn.Linear(in_features, out_features)
        self.base.weight.requires_grad = False
        self.base.bias.requires_grad = False

        self.lora_a = nn.Linear(in_features, rank, bias=False)
        self.lora_b = nn.Linear(rank, out_features, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scaling = alpha / rank

        nn.init.kaiming_uniform_(self.lora_a.weight, a=5 ** 0.5)
        nn.init.zeros_(self.lora_b.weight)

    def forward(self, x):
        base_output = self.base(x)
        # [TODO 1] LoRA adapter branch 출력을 계산하세요.
        # 힌트: dropout -> lora_a -> lora_b -> scaling 순서로 적용합니다.
        dropped_x = self.dropout(x)
        low_rank_features = self.lora_a(dropped_x)
        adapter_logits = self.lora_b(low_rank_features)
        adapter_output = adapter_logits * self.scaling
        return base_output + adapter_output

---
## 4. adapter만 학습되는 모델 구성

### trainable parameter 비율 확인하기

PEFT의 핵심은 “모델을 작게 만든다”가 아니라 “업데이트해야 하는 파라미터를 작게 제한한다”는 데 있습니다. 따라서 모델을 만든 직후 어떤 파라미터가 학습 대상인지 확인해야 합니다.

아래 분류기는 LoRA가 적용된 Linear 레이어를 사용합니다. base Linear는 동결되고, `lora_a`와 `lora_b`만 optimizer에 들어가야 합니다. 이 구조가 맞아야 학습 후 저장해야 할 adapter도 작게 유지됩니다.

In [7]:
class LoRAClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.classifier = LoRALinear(
            config['input_dim'],
            config['num_labels'],
            rank=config['rank'],
            alpha=config['alpha'],
            dropout=config['dropout'],
        )

    def forward(self, x):
        return self.classifier(x)

### 학습 가능한 파라미터 확인하기

전체 파라미터 수와 학습 가능한 파라미터 수를 분리해서 출력합니다. 이름, shape, `requires_grad` 상태를 함께 보면 base weight가 의도대로 고정됐는지 바로 확인할 수 있습니다.

모델 규모가 커지면 이런 점검은 더 중요해집니다. target module 이름을 잘못 지정하거나 freeze 처리를 빼먹으면 전체 모델이 학습 대상에 들어갈 수 있고, 그 순간 PEFT 실험의 메모리와 저장 비용 이점이 사라집니다. 작은 모델에서 먼저 확인하는 이유는 같은 습관을 대형 모델에도 그대로 적용하기 위해서입니다.

In [8]:
def parameter_summary(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        'total': total,
        'trainable': trainable,
        'trainable_ratio': trainable / total,
    }

model = LoRAClassifier(config)
summary = parameter_summary(model)
print('파라미터 요약:', summary)
for name, param in model.named_parameters():
    print('파라미터 정보:', name, tuple(param.shape), '학습 여부=', param.requires_grad)

파라미터 요약: {'total': 183, 'trainable': 108, 'trainable_ratio': 0.5901639344262295}
파라미터 정보: classifier.base.weight (3, 24) 학습 여부= False
파라미터 정보: classifier.base.bias (3,) 학습 여부= False
파라미터 정보: classifier.lora_a.weight (4, 24) 학습 여부= True
파라미터 정보: classifier.lora_b.weight (3, 4) 학습 여부= True


### base weight 동결 상태를 따로 점검하기

파라미터 수 요약만으로는 어떤 그룹이 실제로 동결됐는지 놓칠 수 있습니다. 이름에 `base`가 들어간 파라미터와 `lora`가 들어간 파라미터를 나누어 확인하면 구조가 더 명확해집니다.

base 쪽에 학습 가능한 파라미터가 남아 있다면 adapter 학습이 아니라 부분 또는 전체 파인튜닝에 가까워집니다. 반대로 LoRA 쪽 파라미터가 모두 학습 가능해야 adapter가 태스크에 맞게 변할 수 있습니다.

In [9]:
base_trainable = [name for name, param in model.named_parameters() if 'base' in name and param.requires_grad]
lora_trainable = [name for name, param in model.named_parameters() if 'lora' in name and param.requires_grad]

print('학습되는 base 파라미터:', base_trainable)
print('학습되는 LoRA 파라미터:', lora_trainable)

학습되는 base 파라미터: []
학습되는 LoRA 파라미터: ['classifier.lora_a.weight', 'classifier.lora_b.weight']


---
## 5. adapter 학습 실행

### 작은 adapter만 업데이트하는 학습 루프

optimizer는 `requires_grad=True`인 파라미터만 받도록 구성합니다. 이렇게 해야 base weight가 실수로 업데이트되는 일을 막고, PEFT 실험의 의도도 분명해집니다.

LoRA의 `lora_b`를 0으로 초기화하면 처음에는 base 모델과 같은 출력을 내다가 학습이 진행되며 adapter branch가 점차 영향을 주게 됩니다. 이 방식은 기존 모델의 동작을 갑자기 크게 흔들지 않으면서 태스크에 필요한 변화만 추가하는 데 유리합니다.

In [10]:
base_weight_before = model.classifier.base.weight.detach().clone()

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=config['learning_rate'])

for epoch in range(config['epochs']):
    optimizer.zero_grad()
    logits = model(x_train)
    loss = F.cross_entropy(logits, y_train)
    loss.backward()
    optimizer.step()
    print(f'학습 epoch={epoch} 손실={loss.item():.4f}')

학습 epoch=0 손실=1.2084
학습 epoch=1 손실=1.1727
학습 epoch=2 손실=1.0718
학습 epoch=3 손실=0.9233
학습 epoch=4 손실=0.7638
학습 epoch=5 손실=0.5970
학습 epoch=6 손실=0.5283
학습 epoch=7 손실=0.4460


### adapter 학습 결과 확인하기

학습된 모델로 예측 클래스를 만들고 실제 라벨과 비교합니다. 높은 정확도를 목표로 한 데이터셋은 아니므로, 결과는 adapter가 출력에 영향을 주기 시작했다는 확인 정도로 해석하면 됩니다.

성능을 제대로 평가하려면 별도의 validation set과 반복 실험이 필요합니다. 여기서는 구조 실습이므로 학습 전후의 흐름과 trainable parameter 제한이 더 중요한 관찰 포인트입니다.

In [11]:
with torch.no_grad():
    preds = torch.argmax(model(x_train), dim=-1)
    accuracy = (preds == y_train).float().mean().item()

print('학습 데이터 정확도:', round(accuracy, 4))

학습 데이터 정확도: 0.8438


### 학습 후 base weight가 유지되었는지 확인하기

학습 전 저장해 둔 base weight와 학습 후 base weight를 비교합니다. 값이 그대로라면 optimizer가 LoRA 파라미터만 업데이트했다는 뜻입니다.

이 검사는 PEFT 구현에서 흔히 발생하는 실수를 잡아 줍니다. 학습 대상 필터링을 잘못하면 loss는 잘 내려갈 수 있지만, 실제로는 base 모델까지 바뀌어 adapter만 저장하는 전략을 사용할 수 없게 됩니다. 실험 결과를 믿기 전에 이런 구조적 조건을 먼저 확인해야 합니다.

In [12]:
base_weight_after = model.classifier.base.weight.detach()
base_weight_unchanged = torch.allclose(base_weight_before, base_weight_after)
print('학습 후 base weight 유지 여부:', base_weight_unchanged)

학습 후 base weight 유지 여부: True
